**Submitted by:** Ansh Upadhyay

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
from transformers import AutoTokenizer, AutoModelForMaskedLM

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

C:\Users\anshu\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


In [2]:
def bpe_tokenize_word(word, merges, base_vocab):
    # Step 1: split the word into letters (unknown letters become [UNK])
    tokens = []
    for ch in word:
        if ch in base_vocab:
            tokens.append(ch)
        else:
            tokens.append("[UNK]")

    # Step 2: apply each merge rule in order: join every pair (a, b) into one token
    for (a, b), merged in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i + 1 < len(tokens) and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(merged)
                i = i + 2
            else:
                new_tokens.append(tokens[i])
                i = i + 1
        tokens = new_tokens

    return tokens


merges = [(("u", "g"), "ug"), (("u", "n"), "un"), (("h", "ug"), "hug")]
base_vocab = set("bghnpsu")

for word in ["hugs", "bug", "pun", "mug", "thug", "unhug"]:
    print(word, "->", bpe_tokenize_word(word, merges, base_vocab))

hugs -> ['hug', 's']
bug -> ['b', 'ug']
pun -> ['p', 'un']
mug -> ['[UNK]', 'ug']
thug -> ['[UNK]', 'hug']
unhug -> ['un', 'hug']


In [3]:
def wordpiece_tokenize_word(word, vocab):
    tokens = []
    start = 0
    while start < len(word):
        found = None
        # try the longest piece first, then make it shorter
        for end in range(len(word), start, -1):
            piece = word[start:end]
            if start > 0:
                piece = "##" + piece          # pieces inside a word start with ##
            if piece in vocab:
                found = piece
                break
        if found is None:                     # nothing matched: whole word is unknown
            return ["[UNK]"]
        tokens.append(found)
        start = end                           # continue after the matched piece
    return tokens


vocab = {"[UNK]", "b", "h", "p", "u", "##u", "##g", "##n", "##s", "hu", "hug", "##gs", "##un"}

for word in ["hugs", "bun", "pugs", "hu", "mug", "hugsx"]:
    print(word, "->", wordpiece_tokenize_word(word, vocab))

hugs -> ['hug', '##s']
bun -> ['b', '##un']
pugs -> ['p', '##u', '##gs']
hu -> ['hu']
mug -> ['[UNK]']
hugsx -> ['[UNK]']


In [4]:
# Load WikiText-2 and keep only real paragraphs (no empty lines, no "= Heading =" lines)
wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

def clean(lines):
    result = []
    for line in lines:
        line = line.strip()
        if line != "" and not line.startswith("="):
            result.append(line)
    return result

train_texts = clean(wiki["train"]["text"])
val_texts = clean(wiki["validation"]["text"])


def train_tokenizer(kind, vocab_size):
    if kind == "bpe":
        tok = Tokenizer(models.BPE(unk_token="[UNK]"))
        trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"])
    else:
        tok = Tokenizer(models.WordPiece(unk_token="[UNK]"))
        trainer = trainers.WordPieceTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"])
    tok.normalizer = normalizers.Lowercase()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.train_from_iterator(train_texts, trainer=trainer)    # train on training text only
    return tok


def subword_stats(tokenizer, texts):
    n_words = 0
    n_tokens = 0
    n_split = 0
    n_unk = 0
    for enc in tokenizer.encode_batch(texts):
        # count how many tokens each word was split into
        count = {}
        for word_id in enc.word_ids:
            if word_id is not None:
                count[word_id] = count.get(word_id, 0) + 1

        for c in count.values():
            n_words += 1
            n_tokens += c
            if c > 1:
                n_split += 1
        n_unk += enc.tokens.count("[UNK]")

    return {"fertility": n_tokens / n_words,
            "split_rate": n_split / n_words,
            "unk_rate": n_unk / n_tokens}


for kind in ["bpe", "wordpiece"]:
    for size in [1000, 4000, 16000]:
        tok = train_tokenizer(kind, size)
        stats = subword_stats(tok, val_texts)                 # measure on validation text
        print(kind, size, "fertility =", round(stats["fertility"], 3),
              "split_rate =", round(stats["split_rate"], 3),
              "| photosynthesis ->", tok.encode("photosynthesis").tokens)

bpe 1000 fertility = 2.861 split_rate = 0.601 | photosynthesis -> ['p', 'h', 'o', 'to', 's', 'y', 'n', 'the', 's', 'is']


bpe 4000 fertility = 1.407 split_rate = 0.258 | photosynthesis -> ['photo', 'syn', 'the', 'sis']


bpe 16000 fertility = 1.129 split_rate = 0.092 | photosynthesis -> ['photo', 'synthesis']


wordpiece 1000 fertility = 4.3 split_rate = 0.835 | photosynthesis -> ['p', '##h', '##o', '##t', '##o', '##s', '##y', '##n', '##t', '##h', '##e', '##s', '##i', '##s']


wordpiece 4000 fertility = 1.526 split_rate = 0.309 | photosynthesis -> ['ph', '##ot', '##os', '##yn', '##th', '##es', '##is']


wordpiece 16000 fertility = 1.151 split_rate = 0.102 | photosynthesis -> ['photos', '##yn', '##the', '##sis']


In [5]:
def tokens_per_word(tokenizer, text):
    n_tokens = len(tokenizer.tokenize(text))
    n_words = len(text.split())
    return n_tokens / n_words


names = ["google-bert/bert-base-uncased",
         "google-bert/bert-base-multilingual-cased",
         "openai-community/gpt2"]

sentences = {"English":  "I like Indian food very much.",
             "Hindi":    "मुझे भारतीय खाना बहुत पसंद है।",
             "Hinglish": "Mujhe Indian khana bahut pasand hai."}

for name in names:
    tok = AutoTokenizer.from_pretrained(name)
    print(name)
    for lang, text in sentences.items():
        print("   ", lang, round(tokens_per_word(tok, text), 2), "tokens per word")

# What does bert-base-uncased do to the Hindi sentence?
tok = AutoTokenizer.from_pretrained(names[0])
ids = tok.encode(sentences["Hindi"], add_special_tokens=False)
print("\nOriginal :", sentences["Hindi"])
print("Decoded  :", tok.decode(ids))

C:\Users\anshu\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


google-bert/bert-base-uncased
    English 1.17 tokens per word
    Hindi 3.17 tokens per word
    Hinglish 2.17 tokens per word


google-bert/bert-base-multilingual-cased
    English 1.17 tokens per word
    Hindi 2.17 tokens per word
    Hinglish 2.0 tokens per word


openai-community/gpt2
    English 1.17 tokens per word
    Hindi 7.83 tokens per word
    Hinglish 2.33 tokens per word



Original : मुझे भारतीय खाना बहुत पसंद है।
Decoded  : [UNK] भारतीय खाना बहत पसद ह ।


In [6]:
def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.shape[-1]
    # Step 1: how similar is each query to each key
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
    # Step 2: blocked positions (mask == False) get -infinity
    if mask is not None:
        scores = scores.masked_fill(mask == False, float("-inf"))
    # Step 3: softmax -> weights (each row adds up to 1), then mix the values
    weights = F.softmax(scores, dim=-1)
    output = weights @ v
    return output, weights


def make_padding_mask(input_ids, pad_id):
    B, T = input_ids.shape
    mask = input_ids != pad_id            # True for real tokens, False for padding
    return mask.view(B, 1, 1, T)          # extra dims so it fits (B, H, T, T)


# Test: 2 sentences of 5 tokens, the second one ends with 2 padding tokens (id 0)
ids = torch.tensor([[5, 6, 7, 8, 9],
                    [5, 6, 7, 0, 0]])
mask = make_padding_mask(ids, pad_id=0)

q = torch.randn(2, 3, 5, 4)               # (batch, heads, tokens, d_k)
k = torch.randn(2, 3, 5, 4)
v = torch.randn(2, 3, 5, 4)

out, w = scaled_dot_product_attention(q, k, v, mask)
torch_out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)

print("output shape :", out.shape)
print("weights shape:", w.shape)
print("same as PyTorch:", torch.allclose(out, torch_out, atol=1e-5))
print("weights of sentence 2 (last 2 columns are padding):")
print(w[1, 0].round(decimals=2))

output shape : torch.Size([2, 3, 5, 4])
weights shape: torch.Size([2, 3, 5, 5])
same as PyTorch: True
weights of sentence 2 (last 2 columns are padding):
tensor([[0.3300, 0.1000, 0.5800, 0.0000, 0.0000],
        [0.1100, 0.5300, 0.3600, 0.0000, 0.0000],
        [0.2600, 0.1600, 0.5800, 0.0000, 0.0000],
        [0.3200, 0.3200, 0.3600, 0.0000, 0.0000],
        [0.5100, 0.0800, 0.4100, 0.0000, 0.0000]])


In [7]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):                 # (B, T, D) -> (B, H, T, D/H)
        B, T, D = x.shape
        return x.view(B, T, self.n_heads, D // self.n_heads).transpose(1, 2)

    def forward(self, x, mask):
        q = self.split_heads(self.W_q(x))
        k = self.split_heads(self.W_k(x))
        v = self.split_heads(self.W_v(x))
        out, _ = scaled_dot_product_attention(q, k, v, mask)
        B, H, T, d = out.shape
        out = out.transpose(1, 2).reshape(B, T, H * d)    # join the heads back
        return self.W_o(out)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = SelfAttention(d_model, n_heads)
        self.ff = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        x = self.norm1(x + self.attn(x, mask))    # attention + residual
        x = self.norm2(x + self.ff(x))            # feed-forward + residual
        return x


class TinyEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=32, n_heads=4, n_layers=2, max_len=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads) for _ in range(n_layers)])

    def forward(self, ids, mask=None):
        positions = torch.arange(ids.shape[1])
        x = self.token_emb(ids) + self.pos_emb(positions)
        for layer in self.layers:
            x = layer(x, mask)
        return x

In [8]:
@torch.no_grad()
def padding_leakage(model, ids, junk_ids, pad_id, use_mask=True):
    # use the SAME mask for both runs
    if use_mask:
        mask = make_padding_mask(ids, pad_id)
    else:
        mask = None

    out1 = model(ids, mask)
    out2 = model(junk_ids, mask)

    real = ids != pad_id                       # only compare the real tokens
    return (out1[real] - out2[real]).abs().max().item()


torch.manual_seed(42)
encoder = TinyEncoder(vocab_size=50).eval()

ids      = torch.tensor([[7, 8, 9, 0, 0, 0], [10, 11, 12, 13, 14, 0]])
junk_ids = torch.tensor([[7, 8, 9, 31, 32, 33], [10, 11, 12, 13, 14, 34]])   # only padding changed

print("with mask   :", padding_leakage(encoder, ids, junk_ids, pad_id=0, use_mask=True))
print("without mask:", padding_leakage(encoder, ids, junk_ids, pad_id=0, use_mask=False))

with mask   : 0.0
without mask: 0.5616906881332397


In [9]:
bert_tok = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
bert_mlm = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased").to(device).eval()

Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [10]:
def describe_bert_input(enc, tokenizer):
    ids = enc["input_ids"][0].tolist()
    types = enc["token_type_ids"][0].tolist()
    attn = enc["attention_mask"][0].tolist()

    cls_position = ids.index(tokenizer.cls_token_id)
    sep_positions = [i for i in range(len(ids)) if ids[i] == tokenizer.sep_token_id]
    segment_b_tokens = sum(1 for i in range(len(ids)) if types[i] == 1 and attn[i] == 1)
    pad_tokens = attn.count(0)

    return {"cls_position": cls_position, "sep_positions": sep_positions,
            "segment_b_tokens": segment_b_tokens, "pad_tokens": pad_tokens}


pair = bert_tok("The cat sat on the mat.", "It looked very comfortable.",
                padding="max_length", max_length=20, return_tensors="pt")

tokens = bert_tok.convert_ids_to_tokens(pair["input_ids"][0])
for i in range(len(tokens)):
    print(i, tokens[i], "type =", pair["token_type_ids"][0][i].item(), "attn =", pair["attention_mask"][0][i].item())

print(describe_bert_input(pair, bert_tok))

0 [CLS] type = 0 attn = 1
1 the type = 0 attn = 1
2 cat type = 0 attn = 1
3 sat type = 0 attn = 1
4 on type = 0 attn = 1
5 the type = 0 attn = 1
6 mat type = 0 attn = 1
7 . type = 0 attn = 1
8 [SEP] type = 0 attn = 1
9 it type = 1 attn = 1
10 looked type = 1 attn = 1
11 very type = 1 attn = 1
12 comfortable type = 1 attn = 1
13 . type = 1 attn = 1
14 [SEP] type = 1 attn = 1
15 [PAD] type = 0 attn = 0
16 [PAD] type = 0 attn = 0
17 [PAD] type = 0 attn = 0
18 [PAD] type = 0 attn = 0
19 [PAD] type = 0 attn = 0
{'cls_position': 0, 'sep_positions': [8, 14], 'segment_b_tokens': 6, 'pad_tokens': 5}


In [11]:
def mask_tokens(input_ids, special_mask, mask_token_id, vocab_size, mlm_prob=0.15, generator=None):
    masked_ids = input_ids.clone()      # copy, so the original is not changed
    labels = input_ids.clone()
    shape = input_ids.shape

    # Step 1: choose 15% of the tokens, but never special tokens or padding
    chosen = torch.rand(shape, generator=generator) < mlm_prob
    chosen = chosen & (special_mask == False)

    # Step 2: tokens that were not chosen get label -100 (ignored by the loss)
    labels[chosen == False] = -100

    # Step 3: 80% of the chosen tokens -> [MASK]
    to_mask = chosen & (torch.rand(shape, generator=generator) < 0.8)
    masked_ids[to_mask] = mask_token_id

    # Step 4: of the other 20%, half -> random word, half stays the same
    to_random = chosen & (to_mask == False) & (torch.rand(shape, generator=generator) < 0.5)
    random_words = torch.randint(vocab_size, shape, generator=generator)
    masked_ids[to_random] = random_words[to_random]

    return masked_ids, labels


batch = bert_tok(val_texts[:512], padding=True, truncation=True, max_length=128,
                 return_tensors="pt", return_special_tokens_mask=True)
original = batch["input_ids"]
special_mask = (batch["special_tokens_mask"] == 1) | (batch["attention_mask"] == 0)

masked_ids, labels = mask_tokens(original, special_mask, bert_tok.mask_token_id,
                                 bert_tok.vocab_size, generator=torch.Generator().manual_seed(42))

# Check the percentages
chosen = labels != -100
is_mask = chosen & (masked_ids == bert_tok.mask_token_id)
is_same = chosen & (masked_ids == original)
is_random = chosen & (is_mask == False) & (is_same == False)

n_chosen = chosen.sum().item()
print("chosen   :", round(n_chosen / (special_mask == False).sum().item(), 3), "(target 0.15)")
print("[MASK]   :", round(is_mask.sum().item() / n_chosen, 3), "(target 0.8)")
print("random   :", round(is_random.sum().item() / n_chosen, 3), "(target 0.1)")
print("unchanged:", round(is_same.sum().item() / n_chosen, 3), "(target 0.1)")

chosen   : 0.148 (target 0.15)
[MASK]   : 0.792 (target 0.8)
random   : 0.105 (target 0.1)
unchanged: 0.103 (target 0.1)


In [12]:
@torch.no_grad()
def mask_probs(text):
    """Probability of every word in the vocabulary at the [MASK] position."""
    enc = bert_tok(text, return_tensors="pt").to(device)
    logits = bert_mlm(**enc).logits[0]                         # (tokens, vocab)
    ids = enc["input_ids"][0].tolist()
    mask_pos = ids.index(bert_tok.mask_token_id)
    return F.softmax(logits[mask_pos], dim=-1)


def top_k_predictions(text, k=5):
    probs = mask_probs(text)
    top = torch.topk(probs, k)
    words = bert_tok.convert_ids_to_tokens(top.indices.tolist())
    return [(words[i], round(top.values[i].item(), 3)) for i in range(k)]


sentences = ["The cat sat on the [MASK].",
             "Paris is the capital of [MASK].",
             "She opened the [MASK] and walked into the room.",
             "The doctor gave me some [MASK] for my headache.",
             "Bennett University is located in [MASK]."]
for s in sentences:
    print(s, top_k_predictions(s))

The cat sat on the [MASK]. [('floor', 0.314), ('bed', 0.119), ('couch', 0.107), ('sofa', 0.06), ('ground', 0.055)]
Paris is the capital of [MASK]. [('france', 0.951), ('algeria', 0.012), ('morocco', 0.003), ('reunion', 0.003), ('haiti', 0.003)]
She opened the [MASK] and walked into the room. [('door', 0.987), ('doors', 0.003), ('window', 0.001), ('curtain', 0.001), ('curtains', 0.001)]
The doctor gave me some [MASK] for my headache. [('medication', 0.274), ('medicine', 0.088), ('treatment', 0.079), ('help', 0.075), ('water', 0.072)]
Bennett University is located in [MASK]. [('bennett', 0.185), ('omaha', 0.035), ('sacramento', 0.031), ('burlington', 0.026), ('nashville', 0.023)]


In [13]:
# Run BERT on the masked batch (64 paragraphs at a time) and keep its best guess for every token
predictions = []
with torch.no_grad():
    for start in range(0, len(masked_ids), 64):
        ids = masked_ids[start:start + 64].to(device)
        attn = batch["attention_mask"][start:start + 64].to(device)
        logits = bert_mlm(input_ids=ids, attention_mask=attn).logits
        predictions.append(logits.argmax(dim=-1).cpu())
predictions = torch.cat(predictions)

# Accuracy = how often the guess equals the original word
for name, positions in [("[MASK]", is_mask), ("random", is_random), ("unchanged", is_same)]:
    correct = (predictions[positions] == original[positions]).float().mean().item()
    print(name, "accuracy =", round(correct, 3))

[MASK] accuracy = 0.612
random accuracy = 0.477
unchanged accuracy = 0.965


In [14]:
def pronoun_probs(sentence):
    probs = mask_probs(sentence)
    p_he = probs[bert_tok.convert_tokens_to_ids("he")].item()
    p_she = probs[bert_tok.convert_tokens_to_ids("she")].item()
    return p_he, p_she


occupations = ["nurse", "receptionist", "teacher", "doctor", "engineer", "pilot", "programmer", "scientist"]
templates = ["the {occ} said that [MASK] was tired.",
             "after the meeting, the {occ} explained why [MASK] had to leave early."]

for template in templates:
    print(template)
    for occ in occupations:
        p_he, p_she = pronoun_probs(template.format(occ=occ))
        print("   ", occ, " P(he) =", round(p_he, 3), " P(she) =", round(p_she, 3),
              " log(he/she) =", round(math.log(p_he / p_she), 2))

the {occ} said that [MASK] was tired.
    nurse  P(he) = 0.187  P(she) = 0.637  log(he/she) = -1.22
    receptionist  P(he) = 0.075  P(she) = 0.815  log(he/she) = -2.39
    teacher  P(he) = 0.467  P(she) = 0.377  log(he/she) = 0.21
    doctor  P(he) = 0.48  P(she) = 0.328  log(he/she) = 0.38
    engineer  P(he) = 0.82  P(she) = 0.053  log(he/she) = 2.75
    pilot  P(he) = 0.855  P(she) = 0.081  log(he/she) = 2.36
    programmer  P(he) = 0.788  P(she) = 0.111  log(he/she) = 1.96
    scientist  P(he) = 0.691  P(she) = 0.142  log(he/she) = 1.58
after the meeting, the {occ} explained why [MASK] had to leave early.
    nurse  P(he) = 0.085  P(she) = 0.355  log(he/she) = -1.42
    receptionist  P(he) = 0.102  P(she) = 0.43  log(he/she) = -1.44
    teacher  P(he) = 0.327  P(she) = 0.273  log(he/she) = 0.18
    doctor  P(he) = 0.534  P(she) = 0.081  log(he/she) = 1.89
    engineer  P(he) = 0.527  P(she) = 0.026  log(he/she) = 3.02
    pilot  P(he) = 0.423  P(she) = 0.032  log(he/she) = 2.59


    programmer  P(he) = 0.631  P(she) = 0.061  log(he/she) = 2.34
    scientist  P(he) = 0.538  P(she) = 0.073  log(he/she) = 2.0
